# Lenta — full data pre-label for CVAT on Colab

This notebook prepares **all new annotation data** for the CVAT flow: raw robot videos, extra photos, and optional custom detector weights from Google Drive. It writes a `cvat_video_frames/` tree where every image has a YOLO `.txt` next to it, then creates a zip to download back to the laptop.

It also auto-fixes camera orientation. Videos can come from different robot/camera mounts, so the notebook samples frames from each scene, tries `none` / `cw` / `ccw` / `180`, scores each orientation with the detector, rotates the scene if needed, and only then writes YOLO labels.

Expected Drive layout:

```text
MyDrive/lenta_full_data/
  videos/                 # .mp4/.mov/.mkv, recursive; one CVAT task per video
  photos/                 # extra .jpg/.png, recursive; grouped into photo scenes
  weights/best.pt         # optional: your new pretrained/fine-tuned checkpoint
  cvat_video_frames/      # generated by this notebook
```

If `weights/best.pt` exists, it is used. Otherwise the notebook downloads the OpenFoodFacts price-tag detector from Hugging Face.


In [1]:
# CELL 0 — mount Drive and configure paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/lenta_ful_data")
VIDEO_DIR = DRIVE_ROOT / "videos"
PHOTO_DIR = DRIVE_ROOT / "photos"
WEIGHTS_PATH = DRIVE_ROOT / "weights" / "best.pt"
OUT_DIR = DRIVE_ROOT / "cvat_video_frames"

# Orientation handling:
#   auto = keep raw frames during slicing, then choose none/cw/ccw/180 with the detector per scene.
#   ccw/cw/180/none = force that pre-rotation during slicing; detector auto-orient can still correct later.
EVERY_SECONDS = 2.0
MIN_DIFF = 0.02
ROTATE = "auto"  # auto | ccw | cw | 180 | none
JPEG_QUALITY = 95

# Detector-driven orientation selection. Raise ORIENTATION_IMPROVEMENT if it rotates too eagerly.
AUTO_ORIENT = True
ORIENTATION_SAMPLE_IMAGES = 5
ORIENTATION_IMGSZ = 960
ORIENTATION_MIN_SCORE = 0.50
ORIENTATION_IMPROVEMENT = 1.15
RESET_LABELS_BEFORE_PRELABEL = False  # set True if you want to relabel everything from scratch

# Recall-first pre-labeling: easier to delete extra boxes in CVAT than draw missed tags.
CONF = 0.05
IOU = 0.50
IMGSZ = 1280
MAX_DET = 300

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
PHOTO_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "weights").mkdir(parents=True, exist_ok=True)
print("Drive root:", DRIVE_ROOT)
print("Put videos in:", VIDEO_DIR)
print("Put extra photos in:", PHOTO_DIR)
print("Optional weights:", WEIGHTS_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/lenta_ful_data
Put videos in: /content/drive/MyDrive/lenta_ful_data/videos
Put extra photos in: /content/drive/MyDrive/lenta_ful_data/photos
Optional weights: /content/drive/MyDrive/lenta_ful_data/weights/best.pt


In [2]:
# CELL 1 — install runtime
!pip -q install ultralytics huggingface_hub hf_xet opencv-python-headless tqdm


In [3]:
# CELL 2 — slice videos into frames; orientation auto-correction happens before pre-labeling
import json, shutil
import cv2
import numpy as np
from tqdm.auto import tqdm

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

ROTATION_CHOICES = {"auto", "none", "cw", "ccw", "180"}
assert ROTATE in ROTATION_CHOICES, f"ROTATE must be one of {ROTATION_CHOICES}, got {ROTATE!r}"

def safe_scene_name(path: Path, root: Path, taken: set[str]) -> str:
    rel = path.relative_to(root)
    parts = [p for p in rel.parts[:-1] if p.lower() not in {"videos", "video"}]
    stem = "__".join(parts + [path.stem]) if parts else path.stem
    stem = "".join(c if c.isalnum() or c in "._-" else "_" for c in stem)
    base, name, i = stem or "scene", stem or "scene", 2
    while name in taken:
        name = f"{base}_{i}"
        i += 1
    taken.add(name)
    return name

def rotate_frame(frame, orientation=None):
    orientation = ROTATE if orientation is None else orientation
    if orientation == "cw":
        return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    if orientation == "ccw":
        return cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if orientation == "180":
        return cv2.rotate(frame, cv2.ROTATE_180)
    # auto/none: keep raw frame here; detector auto-orient resolves it later.
    return frame

def gray_thumb(frame):
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.resize(g, (96, 96), interpolation=cv2.INTER_AREA)

def mean_diff(a, b):
    if a is None or b is None:
        return 1.0
    return float(np.mean(cv2.absdiff(a, b))) / 255.0

OUT_DIR.mkdir(parents=True, exist_ok=True)
videos = sorted(p for p in VIDEO_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
manifest = []
taken = set()

for video in videos:
    scene = safe_scene_name(video, VIDEO_DIR, taken)
    sdir = OUT_DIR / scene
    if sdir.exists():
        shutil.rmtree(sdir)
    sdir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    step = max(1, int(round(fps * EVERY_SECONDS)))
    kept, last_thumb = [], None
    pbar = tqdm(range(0, total, step), desc=scene) if total > 0 else []
    for idx in pbar:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        frame = rotate_frame(frame)
        thumb = gray_thumb(frame)
        if MIN_DIFF > 0 and mean_diff(last_thumb, thumb) < MIN_DIFF:
            continue
        last_thumb = thumb
        name = f"{idx:06d}.jpg"
        cv2.imwrite(str(sdir / name), frame, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
        kept.append(name)
    cap.release()
    manifest.append({"scene": scene, "video": str(video), "kept": len(kept), "frames": kept, "fps": fps, "slice_rotate": ROTATE})
    print(f"{scene}: {len(kept)} frames from {video.name} (slice_rotate={ROTATE})")

(OUT_DIR / "slice_manifest_colab.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("videos:", len(videos), "frames:", sum(m["kept"] for m in manifest), "out:", OUT_DIR)


20260517_195951:   0%|          | 0/9 [00:00<?, ?it/s]

20260517_195951: 9 frames from 20260517_195951.mp4


20260517_202013:   0%|          | 0/14 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [4]:
# CELL 3 — add extra photos as CVAT scenes
import shutil

photos = sorted(p for p in PHOTO_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)
photo_root = OUT_DIR / "photos_extra"
if photo_root.exists():
    shutil.rmtree(photo_root)

if photos:
    # Keep top-level photo folders as separate scenes; flat files go into photos_extra/root.
    for src in photos:
        rel = src.relative_to(PHOTO_DIR)
        scene = rel.parts[0] if len(rel.parts) > 1 else "root"
        scene = "photos_extra__" + "".join(c if c.isalnum() or c in "._-" else "_" for c in scene)
        dst_dir = OUT_DIR / scene
        dst_dir.mkdir(parents=True, exist_ok=True)
        stem = "__".join(rel.with_suffix("").parts)
        stem = "".join(c if c.isalnum() or c in "._-" else "_" for c in stem)
        dst = dst_dir / f"{stem}.jpg"
        img = cv2.imread(str(src))
        if img is None:
            print("skip unreadable:", src)
            continue
        cv2.imwrite(str(dst), img, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
print("extra photos copied:", len(photos))
print("scene dirs:", sorted(p.name for p in OUT_DIR.iterdir() if p.is_dir())[:20])


extra photos copied: 28
scene dirs: ['20260517_195951', '20260517_202013', '20260517_202049', '20260517_202113', '20260517_202309', 'photos_extra__root', 'Копия_25_12-20', 'Копия_25_12-20__1_', 'Копия_25_2-10', 'Копия_26_12-20', 'Копия_26_12-20__1_', 'Копия_26_2-10', 'Копия_43_15', 'Копия_49_5']


In [3]:
# CELL 4 — choose detector weights
from huggingface_hub import hf_hub_download

if WEIGHTS_PATH.exists():
    MODEL_PATH = str(WEIGHTS_PATH)
    print("Using custom weights:", MODEL_PATH)
else:
    MODEL_PATH = hf_hub_download(repo_id="openfoodfacts/price-tag-detection", filename="weights/best.pt")
    print("Using OpenFoodFacts weights:", MODEL_PATH)


Using custom weights: /content/drive/MyDrive/lenta_ful_data/weights/best.pt


In [5]:
# CELL 5 — auto-orient scenes, then continue pre-labeling only missing YOLO .txt files, OOM-safe
from ultralytics import YOLO
import torch, gc, json, math

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ORIENTATIONS = ("none", "cw", "ccw", "180")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
USE_HALF = bool(torch.cuda.is_available())

# Очистить GPU-память перед стартом
torch.cuda.empty_cache()
gc.collect()

net = YOLO(MODEL_PATH)

def rotate_array(img, orientation):
    if orientation == "cw":
        return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    if orientation == "ccw":
        return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if orientation == "180":
        return cv2.rotate(img, cv2.ROTATE_180)
    return img

def sample_paths(paths, n):
    if n <= 0 or len(paths) <= n:
        return paths
    if n == 1:
        return [paths[len(paths) // 2]]
    idxs = sorted(set(round(i * (len(paths) - 1) / (n - 1)) for i in range(n)))
    return [paths[i] for i in idxs]

def orientation_score(frames, orientation):
    score = 0.0
    boxes_total = 0
    with torch.inference_mode():
        for frame in frames:
            results = net.predict(
                source=rotate_array(frame, orientation),
                stream=False,
                verbose=False,
                conf=CONF,
                iou=IOU,
                imgsz=min(IMGSZ, ORIENTATION_IMGSZ),
                device=DEVICE,
                batch=1,
                half=USE_HALF,
                max_det=MAX_DET,
            )
            res = results[0]
            boxes = getattr(res, "boxes", None)
            if boxes is None or not len(boxes):
                continue
            boxes_total += len(boxes)
            confs = getattr(boxes, "conf", None)
            if confs is not None:
                score += float(confs.detach().sum().cpu().item())
            score += 0.05 * len(boxes)
    return score, boxes_total

def choose_orientation(imgs):
    samples = sample_paths(imgs, ORIENTATION_SAMPLE_IMAGES)
    frames = []
    for p in samples:
        img = cv2.imread(str(p))
        if img is not None:
            frames.append(img)
    if not frames:
        return "none", {}

    scores = {}
    for orientation in ORIENTATIONS:
        score, boxes = orientation_score(frames, orientation)
        scores[orientation] = {"score": score, "boxes": boxes}
        torch.cuda.empty_cache(); gc.collect()

    best = max(ORIENTATIONS, key=lambda k: (scores[k]["score"], scores[k]["boxes"]))
    none_score = scores["none"]["score"]
    best_score = scores[best]["score"]
    if best == "none":
        return "none", scores
    if best_score < ORIENTATION_MIN_SCORE:
        return "none", scores
    if best_score < none_score * ORIENTATION_IMPROVEMENT:
        return "none", scores
    return best, scores

def apply_orientation(sdir, imgs, orientation):
    if orientation == "none":
        return 0
    changed = 0
    for img_path in imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = rotate_array(img, orientation)
        cv2.imwrite(str(img_path), img, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
        txt = img_path.with_suffix(".txt")
        if txt.exists():
            txt.unlink()
        changed += 1
    return changed

scene_dirs = [
    d for d in sorted(OUT_DIR.iterdir())
    if d.is_dir() and any(p.suffix.lower() in IMG_EXTS for p in d.iterdir())
]

grand_img = grand_box = grand_boxed = 0
summary = []
orientation_summary = []

for sdir in scene_dirs:
    imgs = sorted(
        p for p in sdir.iterdir()
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )

    if RESET_LABELS_BEFORE_PRELABEL:
        for img_path in imgs:
            txt = img_path.with_suffix(".txt")
            if txt.exists():
                txt.unlink()

    applied_orientation = "none"
    scores = {}
    if AUTO_ORIENT:
        applied_orientation, scores = choose_orientation(imgs)
        if applied_orientation != "none":
            changed = apply_orientation(sdir, imgs, applied_orientation)
            print(f"{sdir.name}: auto orientation -> {applied_orientation}; rotated {changed} images; old txt cleared")
            imgs = sorted(
                p for p in sdir.iterdir()
                if p.is_file() and p.suffix.lower() in IMG_EXTS
            )
        else:
            print(f"{sdir.name}: auto orientation -> none")
    orientation_summary.append({"scene": sdir.name, "applied": applied_orientation, "scores": scores})

    # ВАЖНО: пропускаем те картинки, где .txt уже есть. Если сцену повернули, .txt удалены выше.
    todo = [p for p in imgs if not p.with_suffix(".txt").exists()]

    print(f"
{sdir.name}: всего {len(imgs)}, уже готово {len(imgs) - len(todo)}, осталось {len(todo)}")

    for img_path in todo:
        lines = []
        results = res = boxes = None

        try:
            with torch.inference_mode():
                results = net.predict(
                    source=str(img_path),
                    stream=False,
                    verbose=False,
                    conf=CONF,
                    iou=IOU,
                    imgsz=IMGSZ,
                    device=DEVICE,
                    batch=1,
                    half=USE_HALF,
                    max_det=MAX_DET,
                )

            res = results[0]
            boxes = getattr(res, "boxes", None)

            if boxes is not None and len(boxes):
                xywhn = boxes.xywhn.detach().cpu().tolist()
                for cx, cy, bw, bh in xywhn:
                    if bw > 0 and bh > 0:
                        lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

            # Пишем .txt только если предсказание успешно прошло
            img_path.with_suffix(".txt").write_text(
                "
".join(lines) + ("
" if lines else ""),
                encoding="utf-8"
            )

        except torch.cuda.OutOfMemoryError:
            print("OOM на файле:", img_path)
            print("Уменьши IMGSZ до 768, перезапусти runtime и запусти эту ячейку снова.")
            torch.cuda.empty_cache()
            gc.collect()
            raise

        finally:
            del results, res, boxes
            torch.cuda.empty_cache()
            gc.collect()

    # Пересчитываем статистику по всем txt в сцене, включая старые
    n_box = 0
    boxed = 0

    for img_path in imgs:
        txt = img_path.with_suffix(".txt")
        if txt.exists():
            lines = [x for x in txt.read_text(encoding="utf-8").splitlines() if x.strip()]
            n_box += len(lines)
            if lines:
                boxed += 1

    grand_img += len(imgs)
    grand_box += n_box
    grand_boxed += boxed

    summary.append({
        "scene": sdir.name,
        "images": len(imgs),
        "boxed_images": boxed,
        "boxes": n_box,
        "orientation": applied_orientation,
    })

    print(f"{sdir.name}: {len(imgs)} images, {n_box} boxes ({boxed} boxed), orientation={applied_orientation}")

(OUT_DIR / "orientation_manifest_colab.json").write_text(
    json.dumps(orientation_summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
(OUT_DIR / "prelabel_manifest_colab.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"
DONE: {grand_img} images, {grand_box} boxes, {grand_boxed} boxed images")



20260517_195951: всего 9, уже готово 0, осталось 9
20260517_195951: 9 images, 28 boxes (8 boxed)

20260517_202013: всего 12, уже готово 0, осталось 12
20260517_202013: 12 images, 10 boxes (8 boxed)

20260517_202049: всего 8, уже готово 8, осталось 0
20260517_202049: 8 images, 13 boxes (7 boxed)

20260517_202113: всего 13, уже готово 13, осталось 0
20260517_202113: 13 images, 55 boxes (13 boxed)

20260517_202309: всего 9, уже готово 9, осталось 0
20260517_202309: 9 images, 43 boxes (9 boxed)

photos_extra__root: всего 28, уже готово 0, осталось 28
photos_extra__root: 28 images, 745 boxes (28 boxed)

Копия_25_12-20: всего 21, уже готово 0, осталось 21
Копия_25_12-20: 21 images, 213 boxes (21 boxed)

Копия_25_12-20__1_: всего 21, уже готово 0, осталось 21
Копия_25_12-20__1_: 21 images, 253 boxes (21 boxed)

Копия_25_2-10: всего 16, уже готово 0, осталось 16
Копия_25_2-10: 16 images, 380 boxes (16 boxed)

Копия_26_12-20: всего 44, уже готово 0, осталось 44
Копия_26_12-20: 44 images, 377 b

In [6]:
# CELL 6 — verify txt coverage and create a zip for laptop/CVAT
import shutil
from google.colab import files

images = sorted(p for p in OUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)
missing = [p for p in images if not p.with_suffix(".txt").exists()]
print("images:", len(images), "missing txt:", len(missing))
assert not missing[:5], f"Missing txt examples: {missing[:5]}"

zip_base = "/content/cvat_video_frames_prelabelled_full"
shutil.make_archive(zip_base, "zip", OUT_DIR)
print("zip:", zip_base + ".zip")
files.download(zip_base + ".zip")


images: 228 missing txt: 0
zip: /content/cvat_video_frames_prelabelled_full.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Back on the laptop

Unzip the archive as `cvat_video_frames/` in the `worktree-annotation` repo, then run:

```bash
.venv/Scripts/python.exe projects/price_tag_pipeline/scripts/frames_to_cvat.py \
  --frames-dir cvat_video_frames --out cvat_seeds_full
.venv/Scripts/python.exe projects/price_tag_pipeline/scripts/cvat_bootstrap_photos.py \
  --manifest cvat_seeds_full/manifest.json --user admin --password ***
```

Then validate boxes in CVAT, pull them back with `cvat_pull_pack.py`, and run `prepare_data.py` as described in `docs/runbooks/full-data-cvat-colab.md`.
